# Synchronized Alice/Bob training

The notebook calls the same tested workflow as `demo.py`. The two modes differ only in scheduling. Models and token budgets below are editable; teacher and student must share token IDs. Install the examples package with its `notebooks` extra and select the project environment as the kernel.


In [ ]:
from pathlib import Path

from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()

from minisgl_examples.training.parallel_thinking import demo
from minisgl_examples.training.process import isolated

TEACHER = "Qwen/Qwen3-4B"
STUDENT = "Qwen/Qwen3-1.7B"
options = {"synchronized": True, "max_steps": 384}


## Record a teacher trace

`RecordedAsyncLLM` records actions; shared generation helpers align sampled targets to the trace. Prefills and flushes remain masked. The trace records actual cache views at execution, including block publication and clearing. Inspect the answer before choosing the trace as training data; an answer is not guaranteed.


In [ ]:
example = isolated(demo.collect_trace, TEACHER, **options)
teacher = example["rollout"]


In [ ]:
trace, targets = example["trace"], example["targets"]
print("Trace rows:", trace.num_tokens, "Supervised rows:", sum(t != -100 for t in targets))
print("First supervised input → target; recorded position and cache view")
rows = [(token, target) for token, target in zip(trace.tokens, targets) if target != -100]
for token, target in rows[:8]:
    print(token.token_id, "→", target, token.position, token.view)
display(Markdown(teacher["transcript"] + "\n\n" + teacher["answer_text"]))
print("Teacher answer:", teacher["answer"])


## Train and save/reload

The shared training function uses `Trace.to`, `replay_trace` and the scripts’ chunked supervised loss. BF16 master weights and AdamW state stay on CPU, while forward/backward run on the selected accelerator. It checks exact replay logits after checkpoint reload. Use a new output path for each run.


In [ ]:
checkpoint = ROOT / "training_runs/parallel-thinking-sync"
training = isolated(demo.train_student, STUDENT, example, checkpoint, steps=80)
print("Initial:", training["initial"]["loss"], "Final:", training["final"]["loss"])
print("Reload exact:", training["reload_exact"])


## Free generation

The student receives the prompt, but no teacher tokens or publication sequence. No-barrier scheduling is allowed to change its interleaving.


In [ ]:
result = isolated(demo.solve, checkpoint, **options, prompt_ids=teacher["prompt_ids"])
display(Markdown(result["transcript"] + "\n\n" + result["answer_text"]))
print("Student answer:", result["answer"])
print("Same tokens:", all(result[key] == teacher[key]
                          for key in ("history_ids", "current_ids", "answer_ids")))
print("Same publications:", result["merge_events"] == teacher["merge_events"])
